In [3]:
# Bootstrap the MEPS notebook 
# # Allow importing from src/ when this notebook runs inside notebooks/
import sys, os
sys.path.insert(0, os.path.abspath(".."))

print("Working dir:", os.getcwd())

# Confirm we can see the project
from src.config import PROJECT_ROOT, MEPS_SAS7BDAT, BRFSS_XPT
print("MEPS file :", MEPS_SAS7BDAT, "exists:", MEPS_SAS7BDAT.exists())
print("BRFSS file:", BRFSS_XPT, "exists:", BRFSS_XPT.exists())

Working dir: /Users/ht/Desktop/aly6150_diabetes_project/aly6150-diabetes-health-project/notebooks
MEPS file : /Users/ht/Desktop/aly6150_diabetes_project/aly6150-diabetes-health-project/data/raw/meps_2023/h251.sas7bdat exists: True
BRFSS file: /Users/ht/Desktop/aly6150_diabetes_project/aly6150-diabetes-health-project/data/raw/brfss_2023/LLCP2023.XPT exists: True


In [ ]:
# Step 2: Load the MEPS XPT file into a DataFrame

import pandas as pd
import numpy as np
import pyreadstat

# SAS V9 loader — preserves variable labels
meps, meps_meta = pyreadstat.read_sas7bdat(str(MEPS_SAS7BDAT))

print("MEPS shape:", meps.shape)
print("First 10 columns:", meps.columns[:10].tolist())
print("\nDtype summary:")

print(meps.dtypes.value_counts())

MEPS shape: (18919, 1374)
First 10 columns: ['DUID', 'PID', 'DUPERSID', 'PANEL', 'DATAYEAR', 'FAMID31', 'FAMID42', 'FAMID53', 'FAMID23', 'FAMIDYR']

Dtype summary:
float64    1359
str          15
Name: count, dtype: int64


In [4]:
# Variable Inspection 
# Build the search helper 

def search_columns(df, keywords):
    """Return columns whose names contain any of the given keywords (case-insensitive)."""
    keywords = [k.upper() for k in keywords]
    return [col for col in df.columns if any(k in col.upper() for k in keywords)]

In [11]:
# MEPS variable discovery — adapted to confirmed 2023 file structure
# Notes:
#   - No BMI / obesity flag on the Full Year Consolidated file (will come from BRFSS instead)
#   - Comorbidity flags use 4 conditions: HIBPDX, CHDDX, STRKDX, CHOLDX
#   - All year-suffixed variables use "23" for the 2023 file
#   - AGE23X has -1 (inapplicable) codes; filter to 45-64 will exclude them naturally

print("AGE      :", search_columns(meps, ["AGE"]))                                              # expect AGE23X
print("DIAB     :", search_columns(meps, ["DIAB"]))                                             # expect DIABDX_M18, DIABAGED
print("REGION   :", search_columns(meps, ["REGION"]))                                           # expect REGION23 (1=NE, 2=MW, 3=S, 4=W)
print("POV/INC  :", search_columns(meps, ["POVCAT", "POVLEV", "FAMINC", "TTLP"]))              # expect POVCAT23, POVLEV23, FAMINC23, TTLP23X
print("EXP      :", search_columns(meps, ["TOTEXP", "RXEXP", "ERTEXP", "OBVEXP", "IPTEXP",
                                          "OPTEXP", "DVTEXP", "TOTSLF", "RXSLF"]))             # spending by category + OOP
print("UTIL     :", search_columns(meps, ["ERTOT", "OBTOTV", "OPTOTV", "IPDIS",
                                          "IPNGTD", "RXTOT"]))                                  # visit and fill counts
print("INS      :", search_columns(meps, ["INSCOV", "INSURC", "MCDEV", "MCREV",
                                          "PRVEV", "UNINS"]))                                   # coverage type / continuity
print("WT/PSU   :", search_columns(meps, ["PERWT", "VARSTR", "VARPSU"]))                       # complex survey design
print("DEMO     :", search_columns(meps, ["SEX", "RACETHX", "RACEV", "MARRY",
                                          "EDUCYR", "HIDEG"]))                                  # demographics
print("COMORB   :", search_columns(meps, ["HIBPDX", "CHDDX", "STRKDX",
                                          "CHOLDX", "CANCERDX", "ARTHDX",
                                          "ASTHDX", "EMPHDX"]))                                 # 4 core + extras
print("ACCESS   :", search_columns(meps, ["HAVEUS", "DLAYCA", "DLAYPM", "AFRDCA"]))            # usual source of care, delayed care
print("HEALTH   :", search_columns(meps, ["RTHLTH", "MNHLTH", "ADGENH"]))                      # self-rated physical and mental health

AGE      : ['AGE31X', 'AGE42X', 'AGE53X', 'AGE23X', 'AGELAST', 'HIBPAGED', 'CHDAGED', 'ANGIAGED', 'MIAGED', 'OHRTAGED', 'STRKAGED', 'EMPHAGED', 'CHOLAGED', 'DIABAGED', 'ARTHAGED', 'ASTHAGED', 'ADHDAGED', 'WAGEP23X']
DIAB     : ['DIABDX_M18', 'DIABAGED']
REGION   : ['REGION31', 'REGION42', 'REGION53', 'REGION23']
POV/INC  : ['TTLP23X', 'FAMINC23', 'POVCAT23', 'POVLEV23']
EXP      : ['TOTEXP23', 'TOTSLF23', 'OBVEXP23', 'OPTEXP23', 'ERTEXP23', 'IPTEXP23', 'DVTEXP23', 'RXEXP23', 'RXSLF23']
UTIL     : ['OBTOTV23', 'OPTOTV23', 'ERTOT23', 'ERTOTH23', 'IPDIS23', 'IPNGTD23', 'RXTOT23']
INS      : ['PRVEV23', 'MCREV23', 'MCDEV23', 'UNINS23', 'INSCOV23', 'INSURC23']
WT/PSU   : ['PERWT23F', 'VARSTR', 'VARPSU']
DEMO     : ['SEX', 'RACEV1X', 'RACEV2X', 'RACETHX', 'MARRY31X', 'MARRY42X', 'MARRY53X', 'MARRY23X', 'EDUCYR', 'HIDEG', 'PROVSEX42', 'OPSEXP23', 'VISEXP23']
COMORB   : ['HIBPDX', 'CHDDX', 'STRKDX', 'EMPHDX', 'CHOLDX', 'CANCERDX', 'ARTHDX', 'ASTHDX']
ACCESS   : ['HAVEUS42', 'DLAYCA42', 'AFRDCA

In [12]:
# Confirm variable labels for our final analytic key-variable set.

# Notes:
#   - OBESITYDX is NOT in this file. Obesity will be sourced from BRFSS (_BMI5CAT).
#   - Comorbidity index uses 4 core conditions: HIBPDX, CHDDX, STRKDX, CHOLDX.
#     Optional extras (CANCERDX, ARTHDX, ASTHDX) are included for the wider count
#     if they turn out to be present in this file.
#   - Access and self-rated health variables are listed so we can align cross-dataset
#     concepts (BRFSS MEDCOST1 ~ MEPS DLAYCA_M18; BRFSS GENHLTH ~ MEPS RTHLTH53).
#   - Variables marked "(NOT IN FILE)" in the output will be dropped from cleaning.

meps_labels = dict(zip(meps_meta.column_names, meps_meta.column_labels))

key_vars = [
    # --- Identification & sampling design ---
    "DUPERSID", "PANEL", "PERWT23F", "VARSTR", "VARPSU",

    # --- Demographics ---
    "AGE23X", "SEX", "RACETHX", "MARRY23X", "EDUCYR", "HIDEG", "REGION23",

    # --- Income / poverty ---
    "POVCAT23", "POVLEV23", "FAMINC23", "TTLP23X",

    # --- Diabetes ---
    "DIABDX_M18", "DIABAGED",

    # --- Comorbidities (core 4; extras to confirm presence) ---
    "HIBPDX", "CHDDX", "STRKDX", "CHOLDX",
    "CANCERDX", "ARTHDX", "ASTHDX",

    # --- Expenditures (total + by category) ---
    "TOTEXP23", "RXEXP23", "IPTEXP23", "OBVEXP23",
    "OPTEXP23", "ERTEXP23", "DVTEXP23",

    # --- Out-of-pocket ---
    "TOTSLF23", "RXSLF23",

    # --- Utilization (visit / fill / discharge counts) ---
    "OBTOTV23", "OPTOTV23", "ERTOT23", "IPDIS23", "IPNGTD23", "RXTOT23",

    # --- Insurance coverage ---
    "INSCOV23", "INSURC23", "MCDEV23", "MCREV23", "PRVEV23", "UNINS23",

    # --- Access to care ---
    "HAVEUS42", "DLAYCA_M18", "DLAYPM_M18",

    # --- Self-rated health ---
    "RTHLTH53", "MNHLTH53", "ADGENH42",
]

# Print labels grouped visually for easier scanning
print(f"{'Variable':<15} = Label")
print("-" * 70)

for col in key_vars:
    label = meps_labels.get(col, "(NOT IN FILE — drop from cleaning)")
    print(f"{col:<15} = {label}")

# Also print a summary of which ones survived
present = [c for c in key_vars if c in meps_labels]
missing = [c for c in key_vars if c not in meps_labels]

print(f"\n{'-' * 70}")
print(f"Variables PRESENT: {len(present)} of {len(key_vars)}")
print(f"Variables MISSING: {len(missing)}")
if missing:
    print(f"  → Missing variables to drop or replace: {missing}")

Variable        = Label
----------------------------------------------------------------------
DUPERSID        = PERSON ID (DUID + PID)
PANEL           = PANEL NUMBER
PERWT23F        = FINAL PERSON WEIGHT, 2023
VARSTR          = VARIANCE ESTIMATION STRATUM, 2023
VARPSU          = VARIANCE ESTIMATION PSU, 2023
AGE23X          = AGE AS OF 12/31/23 (EDITED/IMPUTED)
SEX             = SEX
RACETHX         = RACE/ETHNICITY (EDITED/IMPUTED)
MARRY23X        = MARITAL STATUS-12/31/23 (EDITED/IMPUTED)
EDUCYR          = YEARS OF EDUC WHEN FIRST ENTERED MEPS
HIDEG           = HIGHEST DEGREE WHEN FIRST ENTERED MEPS
REGION23        = CENSUS REGION AS OF 12/31/23
POVCAT23        = FAMILY INC AS % OF POVERTY LINE - CATEGORICAL
POVLEV23        = FAMILY INC AS % OF POVERTY LINE - CONTINUOUS
FAMINC23        = FAMILY'S TOTAL INCOME
TTLP23X         = PERSON'S TOTAL INCOME
DIABDX_M18      = DIABETES DIAGNOSIS
DIABAGED        = AGE OF DIAGNOSIS-DIABETES
HIBPDX          = HIGH BLOOD PRESSURE DIAG (>17)
CHDDX  

In [13]:
# Inspect raw value distributions for the variables we'll filter or recode in Task 3.

# Goals:
#   1. Confirm coding conventions (especially negative codes for refused/missing).
#   2. See category sizes before applying filters.
#   3. Project the analytic sample size before we commit to the cleaning function.

# Use a small helper to keep the output tidy
def show_dist(series, name, note=""):
    print(f"\n--- {name} ---")
    if note:
        print(f"  ({note})")
    print(series.value_counts(dropna=False).sort_index().to_string())

# ============================================================
# 1. FILTER VARIABLES — these determine the analytic sample
# ============================================================

print("=" * 60)
print("FILTER VARIABLES")
print("=" * 60)

print("\n--- AGE23X (continuous) ---")
print("  (Filter target: 45-64 inclusive)")
print(meps["AGE23X"].describe().to_string())

show_dist(meps["DIABDX_M18"], "DIABDX_M18",
          "Filter target: 1 = Yes; -1/-7/-8 are inapplicable/refused/don't know")

show_dist(meps["REGION23"], "REGION23",
          "1=NE, 2=MW, 3=South, 4=West; -1 = inapplicable")

show_dist(meps["POVCAT23"], "POVCAT23",
          "1=Poor, 2=Near Poor, 3=Low, 4=Middle, 5=High; <200% FPL ≈ 1+2+3")

# ============================================================
# 2. KEY OUTCOME / PREDICTOR VARIABLES — verify they have data
# ============================================================

print("\n" + "=" * 60)
print("OUTCOME & KEY PREDICTORS")
print("=" * 60)

print("\n--- Expenditure variables (continuous, $) ---")
print(meps[["TOTEXP23", "TOTSLF23", "RXEXP23", "ERTEXP23",
            "OBVEXP23", "IPTEXP23"]].describe().to_string())

print("\n--- Utilization counts ---")
print(meps[["OBTOTV23", "ERTOT23", "IPDIS23", "RXTOT23"]].describe().to_string())

show_dist(meps["INSCOV23"], "INSCOV23",
          "1=Any private, 2=Public only, 3=Uninsured")

show_dist(meps["SEX"], "SEX", "1=Male, 2=Female")

show_dist(meps["RACETHX"], "RACETHX",
          "1=Hispanic, 2=NH White, 3=NH Black, 4=NH Asian, 5=NH Other/Multiple")

# ============================================================
# 3. COMORBIDITY FLAGS — check coding conventions
# ============================================================

print("\n" + "=" * 60)
print("COMORBIDITY FLAGS")
print("=" * 60)

for col in ["HIBPDX", "CHDDX", "STRKDX", "CHOLDX"]:
    if col in meps.columns:
        show_dist(meps[col], col,
                  "Expected: 1=Yes, 2=No, -1/-7/-8/-9 = inapplicable/refused")

# ============================================================
# 4. SAMPLE SIZE PROJECTION — what survives each filter?
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE SIZE PROJECTION")
print("=" * 60)

n_total       = len(meps)
n_age         = ((meps["AGE23X"] >= 45) & (meps["AGE23X"] <= 64)).sum()
n_diab        = (meps["DIABDX_M18"] == 1).sum()
n_age_diab    = ((meps["AGE23X"] >= 45) & (meps["AGE23X"] <= 64) &
                 (meps["DIABDX_M18"] == 1)).sum()
n_low_inc     = meps["POVCAT23"].isin([1, 2, 3]).sum()
n_full_filter = ((meps["AGE23X"] >= 45) & (meps["AGE23X"] <= 64) &
                 (meps["DIABDX_M18"] == 1) &
                 meps["POVCAT23"].isin([1, 2, 3])).sum()
n_ne          = ((meps["AGE23X"] >= 45) & (meps["AGE23X"] <= 64) &
                 (meps["DIABDX_M18"] == 1) &
                 (meps["REGION23"] == 1)).sum()
n_ne_low_inc  = ((meps["AGE23X"] >= 45) & (meps["AGE23X"] <= 64) &
                 (meps["DIABDX_M18"] == 1) &
                 (meps["REGION23"] == 1) &
                 meps["POVCAT23"].isin([1, 2, 3])).sum()

print(f"  Total respondents                          : {n_total:>6,}")
print(f"  Age 45-64                                  : {n_age:>6,}")
print(f"  Diagnosed diabetes                         : {n_diab:>6,}")
print(f"  Age 45-64 AND diabetes                     : {n_age_diab:>6,}")
print(f"  <200% FPL (POVCAT 1+2+3)                   : {n_low_inc:>6,}")
print(f"  Age 45-64 + diabetes + low-income          : {n_full_filter:>6,}   ← main MEPS sample")
print(f"  Age 45-64 + diabetes + Northeast           : {n_ne:>6,}")
print(f"  Age 45-64 + diabetes + low-income + NE     : {n_ne_low_inc:>6,}   ← regional slice")

FILTER VARIABLES

--- AGE23X (continuous) ---
  (Filter target: 45-64 inclusive)
count    18919.000000
mean        43.191712
std         24.097835
min         -1.000000
25%         22.000000
50%         44.000000
75%         64.000000
max         85.000000

--- DIABDX_M18 ---
  (Filter target: 1 = Yes; -1/-7/-8 are inapplicable/refused/don't know)
DIABDX_M18
-8.0       18
-7.0       16
-1.0       82
 1.0     2154
 2.0    16649

--- REGION23 ---
  (1=NE, 2=MW, 3=South, 4=West; -1 = inapplicable)
REGION23
-1.0     148
 1.0    2894
 2.0    3838
 3.0    7377
 4.0    4662

--- POVCAT23 ---
  (1=Poor, 2=Near Poor, 3=Low, 4=Middle, 5=High; <200% FPL ≈ 1+2+3)
POVCAT23
1.0    2872
2.0     895
3.0    2485
4.0    5282
5.0    7385

OUTCOME & KEY PREDICTORS

--- Expenditure variables (continuous, $) ---
            TOTEXP23       TOTSLF23        RXEXP23       ERTEXP23       OBVEXP23       IPTEXP23
count   18919.000000   18919.000000   18919.000000   18919.000000   18919.000000   18919.000000
mean  

In [10]:
# Wider search — look for anything related to height, weight, body, or obesity in labels
print("Possible body/weight vars by name:")
print(search_columns(meps, ["WT", "HT", "BODY", "OBES", "ADBMI"]))

print("\nPossible body/weight vars by label:")
for col, label in meps_labels.items():
    upper = label.upper()
    if any(k in upper for k in ["BODY MASS", "BMI", "WEIGHT", "HEIGHT", "OBESITY", "OBESE"]):
        print(f"  {col:20s} = {label}")

Possible body/weight vars by name:
['WHTLGSPK', 'ARTHTYPE', 'CHTHER42', 'CHTHHB42', 'CHTHCO42', 'HHTOTD23', 'OTHTCH23', 'OTHTRI23', 'PERWT23F', 'FAMWT23F', 'FAMWT23C', 'SAQWT23F']

Possible body/weight vars by label:
  PERWT23F             = FINAL PERSON WEIGHT, 2023
  FAMWT23F             = FINAL FAMILY WEIGHT, 2023
  SAQWT23F             = FINAL SAQ PERSON WEIGHT, 2023


In [14]:
%load_ext autoreload
%autoreload 2

from src.meps_cleaning import clean_meps
from src.config import DATA_PROCESSED

print("=" * 60)
print("MEPS CLEANING")
print("=" * 60)

meps_clean = clean_meps(meps)

MEPS CLEANING
  Step 2a — after age 45-64 filter: 4,824 (removed 14,095)
  Step 2b — after diabetes filter:  812 (removed 4,012)

  Final analytic sample: 812 people
  High-spender cutoff (75th pct): $19,741
  Columns: 34


In [15]:
# Sanity checks 

print("Age distribution:")
print(meps_clean["age"].describe().to_string())

print("\nAge band:")
print(meps_clean["age_band"].value_counts().to_string())

print("\nSex:")
print(meps_clean["sex_label"].value_counts(dropna=False).to_string())

print("\nRace/ethnicity:")
print(meps_clean["race_ethnicity"].value_counts(dropna=False).to_string())

print("\nInsurance:")
print(meps_clean["insurance_label"].value_counts(dropna=False).to_string())

print("\nPoverty tier:")
print(meps_clean["poverty_label"].value_counts(dropna=False).to_string())

print("\nNortheast subset size:", meps_clean["is_northeast"].sum())
print("Low-income subset size:", meps_clean["is_low_income"].sum())

print("\nExpenditure summary:")
print(meps_clean[["total_medical_expense", "out_of_pocket_expense",
                  "prescription_expense"]].describe().to_string())

print("\nHigh spender rate:", meps_clean["high_spender"].mean().round(3))

print("\nComorbidity count distribution:")
print(meps_clean["comorbidity_count"].value_counts(dropna=False).sort_index().to_string())

Age distribution:
count    812.000000
mean      56.362069
std        5.308899
min       45.000000
25%       52.000000
50%       57.000000
75%       61.000000
max       64.000000

Age band:
age_band
55-64    522
45-54    290

Sex:
sex_label
Female    417
Male      395

Race/ethnicity:
race_ethnicity
NH White             364
Hispanic             203
NH Black             177
NH Asian              45
NH Other/Multiple     23

Insurance:
insurance_label
Any private    459
Public only    292
Uninsured       61

Poverty tier:
poverty_label
High Income (400%+)         251
Middle Income (200-399%)    231
Poor (<100% FPL)            167
Low Income (125-199%)       118
Near Poor (100-124%)         45

Northeast subset size: 98
Low-income subset size: 330

Expenditure summary:
       total_medical_expense  out_of_pocket_expense  prescription_expense
count             812.000000             812.000000            812.000000
mean            18081.843596            1310.830049           7226.097291
st

In [16]:
# Step 3.4 — Save to disk
from src.config import DATA_PROCESSED

output_path = DATA_PROCESSED / "meps_diabetes_2023_clean.csv"
meps_clean.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Saved: /Users/ht/Desktop/aly6150_diabetes_project/aly6150-diabetes-health-project/data/processed/meps_diabetes_2023_clean.csv
File size: 155.5 KB


In [ ]:
# MEPS Finding Summary

The expenditure distribution is dramatic. Mean total spend = $18,082, median = $8,642, max = $344,782. That's a 40x spread. The standard deviation ($31,793) exceeds the mean, which is the textbook signature of right-skewed healthcare cost data. When we model total_medical_expense in regression, we'll either log-transform it or use a gamma GLM with log link — fitting OLS on raw dollars would be misleading.

Out-of-pocket is meaningful but small relative to total. Median OOP of $425 vs. median total of $8,642 — most spending is covered by insurance, but the OOP tail goes to $44K which is genuinely financially catastrophic for someone in this age band. The OOP/total ratio is a story worth telling.

Prescription spend is huge. Mean Rx of $7,226 represents ~40% of total spend. That's the GLP-1/SGLT-2/insulin reality showing up in the data. RQ3 from your proposal (the pharmacy economics question) was the right call — there's a real story here.

Poverty tier distribution is broader than expected. 31% of this 45–64 diabetic sample are at 400%+ FPL (high income). Diabetes hits across the income spectrum; it's not a "poor person's disease," even if outcomes are worse at lower incomes. Our low-income subset of 330 is the analytic target for inequities analysis, but the full national 812 is the right base for describing diabetes itself.

In [17]:
# Sanity check — read it back
import pandas as pd
meps_reload = pd.read_csv(output_path)
print(f"Reloaded shape: {meps_reload.shape}")
print(f"Columns match: {list(meps_reload.columns) == list(meps_clean.columns)}")

Reloaded shape: (812, 34)
Columns match: True
